# A passagem cega: medir a influência do painel

A referência `open` (o *consenso adjudicado*) é eficiente: o autor adjudica com a maioria do painel já no ecrã. Mas essa é também a sua fragilidade, pois a referência não é *independente* do painel que avalia. Por isso, conduziu-se também uma passagem **cega**: o autor a adjudicar as mesmas revisões apenas a partir do texto, sem o painel à vista. Este notebook mede quanto as duas divergem (a **influência do painel**) e usa a passagem cega como referência de validade independente.

Lê apenas o `gold.csv` (que conserva ambas as passagens) para a parte da influência do painel, e ainda o `annotations.csv` para a parte da calibração, o mesmo zip de exportação que todos os demais notebooks.

In [1]:
%run 00_setup.ipynb

## Carregar a exportação
A referência (o `gold.csv`) é **independente da *run***, uma adjudicação por população partilhada por todas as *runs*, pelo que o `RUN_ID` apenas selecciona as anotações que a calibração cega-face-a-aberta pontua. A medição da influência do painel, em si, não depende dele.

In [2]:
RUN_ID = REFERENCE_RUN   # = 2, a execucao de referencia (a adjudicada); mude para analisar outra
ann    = load("annotations.csv", run=RUN_ID)
gold   = load("gold.csv", run=RUN_ID)
status = load("status.csv", run=RUN_ID)
fail   = load("failures.csv", run=RUN_ID)
dist   = load("pattern_distribution.csv", run=RUN_ID)
sample = load("sample.csv", run=RUN_ID)
meta   = load("meta.csv", run=RUN_ID).iloc[0].to_dict()

print(f"run {RUN_ID}: {meta['label']}  |  painel: {meta['panel']}")
print(f"{len(ann)} linhas de anotacao, {len(gold)} linhas de gold, {len(status)} linhas de estado, {len(fail)} linhas de falha")

run 2: llm_panel #2  |  painel: deepseek/deepseek-v4-flash;meta-llama/llama-3.3-70b-instruct;openai/gpt-oss-120b;qwen/qwen3-next-80b-a3b-instruct
67317 linhas de anotacao, 5000 linhas de gold, 3600 linhas de estado, 57 linhas de falha


## Existirá sequer uma passagem cega?
Primeiro, quantas revisões foram adjudicadas de cada forma, e quantas o foram das **duas** formas. Só a *sobreposição*, ou seja, as revisões feitas em cego **e** em aberto, permite comparar o que é comparável.

In [3]:
b = gold[gold["pass"] == "blind"][["individualId","code","finalLabel"]].rename(columns={"finalLabel":"blind"})
o = gold[gold["pass"] == "open"][["individualId","code","finalLabel"]].rename(columns={"finalLabel":"open"})
print("revisoes adjudicadas, por passagem:")
print(gold.groupby("pass")["individualId"].nunique().to_string())
ov = b.merge(o, on=["individualId","code"], how="inner")
print(f"\nsobreposicao: {ov['individualId'].nunique()} revisoes adjudicadas das DUAS formas -> {len(ov)} celulas partilhadas")

revisoes adjudicadas, por passagem:
pass
blind     35
open     231

sobreposicao: 35 revisoes adjudicadas das DUAS formas -> 629 celulas partilhadas


## A medição da influência do painel
Na sobreposição, com que frequência a decisão do autor **mudou** entre o cego e o aberto? Um valor baixo significa que o painel no ecrã quase não moveu o autor, ou seja, que a cómoda referência `open` é, na prática, tão boa como uma independente. A repartição por direcção diz *em que sentido* o painel influiu: ao vê-lo, terá o autor **acrescentado** um padrão (aberto presente, cego ausente) ou **removido** um (aberto ausente, cego presente)?

In [4]:
flip = ov["blind"] != ov["open"]
print(f"decisoes que mudaram cego <-> aberto: {int(flip.sum())} / {len(ov)} = {100*flip.mean():.2f}%")
print(f"  aberto ACRESCENTOU (aberto presente, cego ausente): {int((~ov['blind'] & ov['open']).sum())}")
print(f"  aberto REMOVEU     (aberto ausente, cego presente): {int(( ov['blind'] & ~ov['open']).sum())}")

decisoes que mudaram cego <-> aberto: 12 / 629 = 1.91%
  aberto ACRESCENTOU (aberto presente, cego ausente): 4
  aberto REMOVEU     (aberto ausente, cego presente): 8


> **Como lê-la.** Esta percentagem *é* a influência do painel, medida directamente. Um valor pequeno é a resposta mais forte à objecção óbvia: «a sua referência viu o painel, logo não é independente». Pode agora dizer-se *quanto* isso importou, em vez de o contornar. (E corta nos dois sentidos: só é tão fiável quanto a sobreposição for ampla, sendo 35 revisões uma primeira leitura, e não a palavra final.)

### Quão estável é esta percentagem?
Com 12 divergências em 629 células, convém delimitar a incerteza: o intervalo de Wilson a 95% para a proporção de divergência. As células de uma mesma avaliação não são independentes, pelo que o intervalo real é algo mais largo — a leitura defensável é a ordem de grandeza, e não o valor exacto.

In [5]:
# intervalo de confianca (Wilson, 95%) da proporcao de divergencia
n, k = len(ov), int(flip.sum())
z = 1.959963984540054
p = k / n
den = 1 + z**2 / n
centro = (p + z**2 / (2*n)) / den
meia = z * ((p*(1-p)/n + z**2/(4*n**2)) ** 0.5) / den
print(f"IC 95% (Wilson) da divergencia: [{100*(centro-meia):.1f}%; {100*(centro+meia):.1f}%]  (k={k}, n={n})")

IC 95% (Wilson) da divergencia: [1.1%; 3.3%]  (k=12, n=629)


## Onde se concentra a discordância
As divergências não se distribuem por igual: alguns padrões são mais sujeitos à influência do painel do que outros. São as famílias em que ver o painel mais alterou o juízo do autor.

In [6]:
pp = ov.assign(flip=flip).groupby("code").agg(cells=("flip","size"), flips=("flip","sum"))
pp["pct"] = (100 * pp["flips"] / pp["cells"]).round(1)
pp[pp["flips"] > 0].sort_values("flips", ascending=False)

,cells,flips,pct
code,,,
PM-1,33,3,9.1
TM-1,33,2,6.1
TM-3,33,2,6.1
PE-2,34,1,2.9
PE-3,33,1,3.0
PM-2,33,1,3.0
PM-4,33,1,3.0
TM-2,33,1,3.0


## Calibração contra a referência independente
O ganho é pontuar os modelos contra a referência **cega**, e não apenas contra a aberta. Face à referência cega, a revocação é uma revocação verdadeira, cada modelo contra um humano independente, sem nada que o painel tenha semeado a favor da referência. Execute ambos e compare o F1 macro.

In [7]:
for p in ["open", "blind"]:
    cd = cells(ann, gold, p)
    print(f"-- F1 macro face ao gold {p}  ({cd['individualId'].nunique()} revisoes) --")
    print(macro(calibration(cd))["f1"].sort_values(ascending=False).to_string(), "\n")

-- F1 macro face ao gold open  (231 revisoes) --
model
qwen/qwen3-next-80b-a3b-instruct     0.446
meta-llama/llama-3.3-70b-instruct    0.272
deepseek/deepseek-v4-flash           0.204
panel-majority                       0.194
openai/gpt-oss-120b                  0.179 

-- F1 macro face ao gold blind  (35 revisoes) --
model
meta-llama/llama-3.3-70b-instruct    0.066
qwen/qwen3-next-80b-a3b-instruct     0.053
openai/gpt-oss-120b                  0.043
deepseek/deepseek-v4-flash           0.035
panel-majority                       0.035 



> **Como lê-la, e a ressalva que mais importa.** A referência cega é hoje pequena (a sobreposição acima), pelo que a sua calibração deve ser tratada como forma, e não como resultado. Um punhado de células positivas por padrão torna cada F1 instável, e uma macro próxima de zero diz sobretudo «positivos insuficientes para pontuar», e não «o modelo não presta». Ou seja, a referência **aberta** sustenta os números de validade em escala, com a percentagem de influência do painel acima como a ressalva que os defende, e a referência **cega** é a verificação independente, que ganhará peso quando a passagem cega for alargada. Esse alargamento é o trabalho futuro com mais retorno.